In [5]:
"""01-01 실습: 태그를 분해하고 공정 단계를 분류합니다."""



import pandas as pd

In [6]:
import os

In [7]:
os.getcwd()

'c:\\ex\\week9\\1일차_실습'

In [8]:
df = pd.read_csv('./01-01_철강_공정_개관_설비태그.csv')

In [10]:
df.columns

Index(['tag', 'unit', 'sample_value', 'note'], dtype='str')

In [9]:
# 태그 형식: 공장-공정-설비-호기-계측 코드

In [12]:
df["tag"].str.split("-", expand=True)

,0,1,2,3,4
0,PL1,SNT,FAN,01,VIB
1,PL1,SNT,FAN,01,CUR
2,PL1,SNT,CNV,02,SPD
3,PL1,CKO,PMP,01,PRS
4,PL1,CKO,FUR,01,TMP
5,PL1,BF,BLW,01,FLW
6,PL1,BF,BLW,01,VIB
7,PL1,BF,FUR,01,PRS
8,PL1,BF,FUR,01,TMP
9,PL1,BOF,CRN,01,CUR


In [13]:
parts=df["tag"].str.split("-", expand=True)

In [14]:
df.index

RangeIndex(start=0, stop=24, step=1)

In [16]:
parts=df.reindex(columns=range(5))

In [17]:
parts.columns = [
    "plant",
    "process_code",
    "equipment_code",
    "unit_no",
    "measurement_code",
]

In [18]:
df = pd.concat([df, parts], axis=1)

In [19]:
df

,tag,unit,sample_value,note,plant,process_code,equipment_code,unit_no,measurement_code
0,PL1-SNT-FAN-01-VIB,mm/s,2.8,소결 주배풍기 진동,NaN,NaN,NaN,NaN,NaN
1,PL1-SNT-FAN-01-CUR,A,412.5,소결 주배풍기 전류,NaN,NaN,NaN,NaN,NaN
2,PL1-SNT-CNV-02-SPD,m/s,1.6,소결광 이송 컨베이어 속도,NaN,NaN,NaN,NaN,NaN
3,PL1-CKO-PMP-01-PRS,bar,6.2,코크스로 냉각수 펌프 토출압,NaN,NaN,NaN,NaN,NaN
4,PL1-CKO-FUR-01-TMP,degC,1042.0,코크스로 연소실 온도,NaN,NaN,NaN,NaN,NaN
5,PL1-BF-BLW-01-FLW,Nm3/min,5200.0,고로 송풍기 풍량,NaN,NaN,NaN,NaN,NaN
6,PL1-BF-BLW-01-VIB,mm/s,3.4,고로 송풍기 진동,NaN,NaN,NaN,NaN,NaN
7,PL1-BF-FUR-01-PRS,kPa,210.0,고로 노정압,NaN,NaN,NaN,NaN,NaN
8,PL1-BF-FUR-01-TMP,degC,1180.0,고로 열풍 온도,NaN,NaN,NaN,NaN,NaN
9,PL1-BOF-CRN-01-CUR,A,305.0,전로 장입 크레인 전류,NaN,NaN,NaN,NaN,NaN


In [22]:
valid_structure = df["tag"].str.count("-").eq(4) & parts.notna().all(axis=1)

In [23]:
process_names = {
    "SNT": "소결",
    "CKO": "코크스",
    "BF": "고로",
    "BOF": "전로",
    "CCM": "연주",
    "HSM": "열연",
    "CRM": "냉연",
    "UTL": "유틸리티",
}

In [24]:
# 이 실습에서는 제선·제강·연주를 상공정, 압연을 하공정으로 분류합니다.
# 전로(BOF)는 용선을 용강으로 만드는 제강 공정이므로 상공정입니다.

In [25]:
stage_by_process = {
    "SNT": "상공정",
    "CKO": "상공정",
    "BF": "상공정",
    "BOF": "상공정",
    "CCM": "상공정",
    "HSM": "하공정",
    "CRM": "하공정",
    "UTL": "유틸리티",
}

In [27]:
df["process_code"].map(process_names).fillna("미등록")

0     미등록
1     미등록
2     미등록
3     미등록
4     미등록
5     미등록
6     미등록
7     미등록
8     미등록
9     미등록
10    미등록
11    미등록
12    미등록
13    미등록
14    미등록
15    미등록
16    미등록
17    미등록
18    미등록
19    미등록
20    미등록
21    미등록
22    미등록
23    미등록
Name: process_code, dtype: str

In [ ]:
df["process_name"] = df["process_code"].map(process_names).fillna("미등록")

In [29]:
df["stage"] = df["process_code"].map(stage_by_process).fillna("미등록")

In [30]:
df

,tag,unit,sample_value,note,plant,process_code,equipment_code,unit_no,measurement_code,stage
0,PL1-SNT-FAN-01-VIB,mm/s,2.8,소결 주배풍기 진동,NaN,NaN,NaN,NaN,NaN,미등록
1,PL1-SNT-FAN-01-CUR,A,412.5,소결 주배풍기 전류,NaN,NaN,NaN,NaN,NaN,미등록
2,PL1-SNT-CNV-02-SPD,m/s,1.6,소결광 이송 컨베이어 속도,NaN,NaN,NaN,NaN,NaN,미등록
3,PL1-CKO-PMP-01-PRS,bar,6.2,코크스로 냉각수 펌프 토출압,NaN,NaN,NaN,NaN,NaN,미등록
4,PL1-CKO-FUR-01-TMP,degC,1042.0,코크스로 연소실 온도,NaN,NaN,NaN,NaN,NaN,미등록
5,PL1-BF-BLW-01-FLW,Nm3/min,5200.0,고로 송풍기 풍량,NaN,NaN,NaN,NaN,NaN,미등록
6,PL1-BF-BLW-01-VIB,mm/s,3.4,고로 송풍기 진동,NaN,NaN,NaN,NaN,NaN,미등록
7,PL1-BF-FUR-01-PRS,kPa,210.0,고로 노정압,NaN,NaN,NaN,NaN,NaN,미등록
8,PL1-BF-FUR-01-TMP,degC,1180.0,고로 열풍 온도,NaN,NaN,NaN,NaN,NaN,미등록
9,PL1-BOF-CRN-01-CUR,A,305.0,전로 장입 크레인 전류,NaN,NaN,NaN,NaN,NaN,미등록


In [31]:
# 정리

In [32]:
df = pd.read_csv('./01-01_철강_공정_개관_설비태그.csv')

In [33]:
df

,tag,unit,sample_value,note
0,PL1-SNT-FAN-01-VIB,mm/s,2.8,소결 주배풍기 진동
1,PL1-SNT-FAN-01-CUR,A,412.5,소결 주배풍기 전류
2,PL1-SNT-CNV-02-SPD,m/s,1.6,소결광 이송 컨베이어 속도
3,PL1-CKO-PMP-01-PRS,bar,6.2,코크스로 냉각수 펌프 토출압
4,PL1-CKO-FUR-01-TMP,degC,1042.0,코크스로 연소실 온도
5,PL1-BF-BLW-01-FLW,Nm3/min,5200.0,고로 송풍기 풍량
6,PL1-BF-BLW-01-VIB,mm/s,3.4,고로 송풍기 진동
7,PL1-BF-FUR-01-PRS,kPa,210.0,고로 노정압
8,PL1-BF-FUR-01-TMP,degC,1180.0,고로 열풍 온도
9,PL1-BOF-CRN-01-CUR,A,305.0,전로 장입 크레인 전류


In [34]:
# 태그 형식: 공장-공정-설비-호기-계측 코드
parts = df["tag"].str.split("-", expand=True)
parts = parts.reindex(columns=range(5))
parts.columns = [
    "plant",
    "process_code",
    "equipment_code",
    "unit_no",
    "measurement_code",
]

In [35]:
df = pd.concat([df, parts], axis=1)
valid_structure = df["tag"].str.count("-").eq(4) & parts.notna().all(axis=1)

In [36]:
process_names = {
    "SNT": "소결",
    "CKO": "코크스",
    "BF": "고로",
    "BOF": "전로",
    "CCM": "연주",
    "HSM": "열연",
    "CRM": "냉연",
    "UTL": "유틸리티",
}

In [37]:
# 이 실습에서는 제선·제강·연주를 상공정, 압연을 하공정으로 분류합니다.
# 전로(BOF)는 용선을 용강으로 만드는 제강 공정이므로 상공정입니다.
stage_by_process = {
    "SNT": "상공정",
    "CKO": "상공정",
    "BF": "상공정",
    "BOF": "상공정",
    "CCM": "상공정",
    "HSM": "하공정",
    "CRM": "하공정",
    "UTL": "유틸리티",
}

In [38]:
df["process_name"] = df["process_code"].map(process_names).fillna("미등록")
df["stage"] = df["process_code"].map(stage_by_process).fillna("미등록")

In [39]:
print("=== 태그 구조 점검 ===")
print("태그 행 수:", len(df))

=== 태그 구조 점검 ===
태그 행 수: 24


In [40]:
print("5개 마디가 아닌 행 수:", int((~valid_structure).sum()))

5개 마디가 아닌 행 수: 0


In [41]:
df["process_name"] == "미등록"

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
Name: process_name, dtype: bool

In [42]:
print(
    "미등록 공정 코드:",
    sorted(set(df.loc[df["process_name"] == "미등록", "process_code"])),
)

미등록 공정 코드: []


In [43]:
print("\n=== 공정별 태그 수 ===")
print(df.groupby(["process_name", "stage"]).size().sort_values(ascending=False))


=== 공정별 태그 수 ===
process_name  stage
고로            상공정      4
열연            하공정      4
소결            상공정      3
냉연            하공정      3
연주            상공정      3
유틸리티          유틸리티     3
전로            상공정      2
코크스           상공정      2
dtype: int64


In [44]:
print("\n=== 계측 코드별 태그 수 ===")
print(df["measurement_code"].value_counts())


=== 계측 코드별 태그 수 ===
measurement_code
TMP    6
CUR    5
PRS    5
VIB    4
FLW    3
SPD    1
Name: count, dtype: int64


In [45]:
print("\n=== 태그별 분류 결과 ===")
print(
    df[
        [
            "tag",
            "plant",
            "process_name",
            "equipment_code",
            "unit_no",
            "measurement_code",
            "stage",
            "unit",
        ]
    ].to_string(index=False)
)



=== 태그별 분류 결과 ===
               tag plant process_name equipment_code unit_no measurement_code stage    unit
PL1-SNT-FAN-01-VIB   PL1           소결            FAN      01              VIB   상공정    mm/s
PL1-SNT-FAN-01-CUR   PL1           소결            FAN      01              CUR   상공정       A
PL1-SNT-CNV-02-SPD   PL1           소결            CNV      02              SPD   상공정     m/s
PL1-CKO-PMP-01-PRS   PL1          코크스            PMP      01              PRS   상공정     bar
PL1-CKO-FUR-01-TMP   PL1          코크스            FUR      01              TMP   상공정    degC
 PL1-BF-BLW-01-FLW   PL1           고로            BLW      01              FLW   상공정 Nm3/min
 PL1-BF-BLW-01-VIB   PL1           고로            BLW      01              VIB   상공정    mm/s
 PL1-BF-FUR-01-PRS   PL1           고로            FUR      01              PRS   상공정     kPa
 PL1-BF-FUR-01-TMP   PL1           고로            FUR      01              TMP   상공정    degC
PL1-BOF-CRN-01-CUR   PL1           전로            CRN      01 